In [2]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.3 MB/s eta 0:00:00


In [6]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="nmmrbjRIkKkRTCm7qfCQ")
project = rf.workspace("bere-gcgue").project("smartphones_capstone")
dataset = project.version(4).download("yolov11")

loading Roboflow workspace...
loading Roboflow project...
Exporting format yolov11 in progress : 45.0%
Version export complete for yolov11 format


Extracting Dataset Version Zip to smartphones_capstone-4 in yolov11:: 100%|██████████| 2235/2235 [00:00<00:00, 7977.00it/s]


In [4]:
import os

print(os.listdir("/content"))

['.config', 'smartphones_capstone-1', 'yolo11s.pt', 'runs', 'sample_data']


In [7]:
from ultralytics import YOLO

model = YOLO("yolo11s.pt")  # small model, good balance for Colab T4

results = model.train(
    data="/content/smartphones_capstone-4/data.yaml",
    epochs=120,
    imgsz=640,
    batch=16,
    cos_lr=True,
    scale=0.5,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    patience=30
)

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/smartphones_capstone-4/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=120, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True,

In [8]:
!ls /content

runs	     smartphones_capstone-1  yolo11s.pt
sample_data  smartphones_capstone-4  yolo26n.pt


In [13]:
# Evaluate model performance on the validation set
metrics = model.val()

precision = metrics.box.mp
recall = metrics.box.mr

f1 = 2 * (precision * recall) / (precision + recall + 1e-9)

print("\n=== FINAL TRAINING METRICS ===")
print(f"mAP50:     {metrics.box.map50 * 100:.2f}%")
print(f"mAP50-95:  {metrics.box.map * 100:.2f}%")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall:    {recall * 100:.2f}%")
print(f"F1-Score:  {f1 * 100:.2f}%")

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 912.6±558.6 MB/s, size: 28.8 KB)
val: Scanning /content/smartphones_capstone-4/valid/labels.cache... 67 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 67/67 14.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.9it/s 2.7s
                   all         67         82      0.928      0.822      0.871      0.858
  asus_rog_phone_9_pro          6          6      0.964      0.833      0.942      0.942
            iphone_16e          6          6          1       0.84      0.995      0.976
     iphone_17_pro_max         10         11      0.712      0.818      0.843      0.837
nothing_cmf_phone_2_pro          5          5      0.963        0.8      0.795      0.795
            oneplus_13          7          9      0.886      0.889      0.895      0.879
      oppo_find_x9_pro        

In [17]:
from ultralytics import YOLO
model = YOLO("/content/runs/detect/train-3/weights/best.pt"
)
model.export(format="onnx", imgsz=640, simplify=True)

Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO11s summary (fused): 101 layers, 9,416,670 parameters, 0 gradients, 21.3 GFLOPs

PyTorch: starting from '/content/runs/detect/train-3/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 14, 8400) (18.3 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 334ms
Prepared 4 packages in 1.54s
Installed 4 packages in 257ms
 + colorama==0.4.6
 + onnx==1.21.0
 + onnxruntime==1.26.0
 + onnxslim==0.1.94

requirements: AutoUpdate success ✅ 2.8s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimmi

'/content/runs/detect/train-3/weights/best.onnx'